## Merge Author Profiles — ORCID Collision

Identifies cases where a single ORCID is claimed by 2+ author profiles and collapses them into one. ORCID is a globally unique identifier — same ORCID on multiple profiles is one of: (a) duplicate splinters (typical Phase 2 ORCID-disjoint output where pre-2025 works stayed on the original profile because of MatchAuthors' `created_date >= 2025-12-20` cutoff), or (b) bad metadata that wrongly attached the same ORCID to multiple distinct people. This notebook merges (a) and excludes (b) via two precision guards.

**Scope:** 207,012 ORCIDs collide on multiple profiles (474,729 profiles total). After guards: ~193,961 groups, ~444,094 profiles, ~250K profiles being collapsed away.

**Precision guards:**
1. **Sink cap.** Drop any group containing a profile with `works_count > 5,000`. The ~60 profiles above this threshold are pollution sinks (e.g., 437K-work "K. Ida", 817K-work "H. Funaba"); merging into them amplifies pollution. Costs 53 groups.
2. **Middle-name contradiction.** Drop any group with 2+ distinct full middle name tokens (length ≥ 3) across its profiles. Catches the same-ORCID-on-different-people class — e.g., Scott *Edward* Baker / Scott *King* Baker / Scott *R.* Baker, where ORCID metadata mis-attributed three real humans to one ID. Costs ~13K groups (~6%).

**Winner selection:** Per group, max `works_count` → oldest `created_date` → lowest `id` (deterministic).

**Loser handling:** Hard `DELETE` from `openalex.authors.openalex_authors`. CreateAuthors recomputes profile aggregates from `work_authors` daily, so the surviving profile picks up the merged works on the next refresh.

**Rollback plan:** No automatic rollback — this is an intentional collapse. The audit log (`openalex.authors.author_merge_log_orcid`) captures `(orcid, winner_author_id, loser_author_id, loser_full_name, loser_works_count, loser_cited_by_count, loser_created_date, merge_timestamp)`, which is sufficient to reconstruct any loser profile if needed (re-create the row from the log; re-attach `work_authors` rows from `STILL_UNMATCHED` via `MatchAuthors`).

### Cell 1: Build merge candidates table

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.merge_candidates_orcid_collision AS
WITH all_orcid_profiles AS (
  SELECT
    orcid, id AS author_id, full_name, display_name, block_key,
    works_count, cited_by_count, created_date,
    COUNT(*)         OVER (PARTITION BY orcid) AS profile_count,
    MAX(works_count) OVER (PARTITION BY orcid) AS group_max_works
  FROM openalex.authors.openalex_authors
  WHERE orcid IS NOT NULL AND orcid != ''
),
post_cap AS (
  -- Guard 1: drop groups containing any sink profile (>5000 works)
  SELECT * FROM all_orcid_profiles
  WHERE profile_count > 1 AND group_max_works <= 5000
),
swapped AS (
  -- Normalize "Last, First Middle" → "First Middle Last" so middle tokens land in the middle
  SELECT *,
    CASE WHEN INSTR(full_name, ',') > 0
      THEN TRIM(SPLIT(full_name, ',', 2)[1]) || ' ' || TRIM(SPLIT(full_name, ',', 2)[0])
      ELSE full_name
    END AS swapped_name
  FROM post_cap
),
tokenized AS (
  SELECT *,
    FILTER(
      TRANSFORM(SPLIT(swapped_name, '\\s+'),
                t -> LOWER(REGEXP_REPLACE(t, '[^A-Za-zÀ-ÿ]', ''))),
      t -> LENGTH(t) > 0
    ) AS tokens
  FROM swapped
),
middles AS (
  -- Middle tokens = everything between first and last, length >= 3 (initials don't count as contradictions)
  SELECT orcid, author_id,
    CASE
      WHEN SIZE(tokens) <= 2 THEN ARRAY()
      ELSE FILTER(SLICE(tokens, 2, SIZE(tokens) - 2), t -> LENGTH(t) >= 3)
    END AS middle_tokens
  FROM tokenized
),
group_middles AS (
  SELECT orcid, ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(middle_tokens))) AS group_middle_tokens
  FROM middles
  GROUP BY orcid
),
clean_groups AS (
  -- Guard 2: drop groups with >1 distinct full middle name token
  SELECT orcid FROM group_middles WHERE SIZE(group_middle_tokens) <= 1
),
ranked AS (
  SELECT pc.*,
    ROW_NUMBER() OVER (
      PARTITION BY pc.orcid
      ORDER BY pc.works_count DESC, pc.created_date ASC, pc.author_id ASC
    ) AS rn
  FROM post_cap pc
  JOIN clean_groups cg ON pc.orcid = cg.orcid
),
winners AS (
  SELECT orcid, author_id AS winner_author_id FROM ranked WHERE rn = 1
)
SELECT
  r.orcid, r.author_id, r.full_name, r.display_name, r.block_key,
  r.works_count, r.cited_by_count, r.created_date, r.profile_count,
  (r.rn = 1) AS is_winner,
  w.winner_author_id
FROM ranked r
JOIN winners w USING (orcid)

### Cell 2: Validation statistics

In [ ]:
SELECT
  COUNT(DISTINCT orcid)                                                         AS total_groups,
  COUNT(*)                                                                      AS total_profiles,
  COUNT(CASE WHEN NOT is_winner THEN 1 END)                                     AS losers_to_delete,
  COUNT(CASE WHEN     is_winner THEN 1 END)                                     AS winners_kept,
  SUM(CASE WHEN NOT is_winner THEN works_count END)                             AS works_to_rewrite,
  COUNT(DISTINCT CASE WHEN profile_count = 2 THEN orcid END)                    AS groups_size_2,
  COUNT(DISTINCT CASE WHEN profile_count = 3 THEN orcid END)                    AS groups_size_3,
  COUNT(DISTINCT CASE WHEN profile_count = 4 THEN orcid END)                    AS groups_size_4,
  COUNT(DISTINCT CASE WHEN profile_count >= 5 THEN orcid END)                   AS groups_size_5plus,
  PERCENTILE_APPROX(works_count, ARRAY(0.5, 0.95, 0.99))                        AS profile_works_pctiles
FROM openalex.authors.merge_candidates_orcid_collision

### Cell 3: Spot-check sample (manual review)

In [ ]:
WITH sampled_orcids AS (
  SELECT DISTINCT orcid FROM openalex.authors.merge_candidates_orcid_collision
  ORDER BY RAND() LIMIT 30
)
SELECT c.orcid, c.author_id, c.full_name, c.block_key,
       c.works_count, c.cited_by_count, c.is_winner, c.winner_author_id, c.profile_count
FROM openalex.authors.merge_candidates_orcid_collision c
JOIN sampled_orcids s ON c.orcid = s.orcid
ORDER BY c.orcid, c.is_winner DESC, c.works_count DESC

### Cell 4: Create audit log

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_merge_log_orcid AS
SELECT
  orcid,
  winner_author_id,
  author_id           AS loser_author_id,
  full_name           AS loser_full_name,
  works_count         AS loser_works_count,
  cited_by_count      AS loser_cited_by_count,
  created_date        AS loser_created_date,
  current_timestamp() AS merge_timestamp
FROM openalex.authors.merge_candidates_orcid_collision
WHERE NOT is_winner

### Cell 5: Snapshot affected work_ids (capture before MERGE rewrites author_id)

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.author_merge_affected_works_orcid AS
SELECT DISTINCT wa.work_id
FROM openalex.works.work_authors wa
JOIN openalex.authors.author_merge_log_orcid log
  ON wa.author_id = log.loser_author_id

### Cell 6: Execute the merge — rewrite work_authors.author_id from losers to winners

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT loser_author_id, winner_author_id
  FROM openalex.authors.author_merge_log_orcid
) AS source
ON target.author_id = source.loser_author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = source.winner_author_id,
    target.updated_at = current_timestamp()

### Cell 7: Delete loser author profiles

In [ ]:
DELETE FROM openalex.authors.openalex_authors
WHERE id IN (SELECT loser_author_id FROM openalex.authors.author_merge_log_orcid)

### Cell 8: Queue affected works for reprocessing by UpdateWorkAuthorships

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.author_merge_affected_works_orcid

### Cell 9: Post-merge verification

In [ ]:
WITH remaining_collisions AS (
  SELECT orcid, COUNT(*) AS profile_count
  FROM openalex.authors.openalex_authors
  WHERE orcid IS NOT NULL AND orcid != ''
  GROUP BY orcid
  HAVING COUNT(*) > 1
)
SELECT
  COUNT(*)                        AS remaining_orcid_collisions,
  COALESCE(SUM(profile_count), 0) AS remaining_colliding_profiles
FROM remaining_collisions